# Session validity check
Pour chaque session : aires disponibles + n neurones par aire, structure des stimuli, lick times.

In [13]:
import glob, os

# ── Scan automatique de data/brut/WR+/ et data/brut/WR-/ ───────────────────
BRUT_DIR = 'data/brut'
GROUPS   = ['WR+', 'WR-']

SESSION_PATHS = []
for grp in GROUPS:
    grp_dir = os.path.join(BRUT_DIR, grp)
    paths   = sorted(glob.glob(os.path.join(grp_dir, '*.nwb')))
    SESSION_PATHS.extend(paths)

print(f'{len(SESSION_PATHS)} fichier(s) .nwb trouve(s) dans {BRUT_DIR}/ :')
for p in SESSION_PATHS:
    grp = os.path.basename(os.path.dirname(p))
    print(f'  [{grp}]  {os.path.basename(p)}')


7 fichier(s) .nwb trouve(s) dans data/brut/ :
  [WR+]  AO026_20181122_180943.nwb
  [WR-]  AO081_20210505_192239.nwb
  [WR-]  AO081_20210506_132701.nwb
  [WR-]  AO081_20210507_131703.nwb
  [WR-]  AO082_20210519_183538.nwb
  [WR-]  AO082_20210520_160848.nwb
  [WR-]  AO082_20210521_164409.nwb


In [14]:
import numpy as np
import pandas as pd
from pynwb import NWBHDF5IO
from collections import Counter

# ── Helpers d'affichage ─────────────────────────────────────────────────────
GRN = '\033[92m'; YLW = '\033[93m'; RED = '\033[91m'; BLD = '\033[1m'; RST = '\033[0m'
HDR = lambda s: print(f'\n{BLD}{s}{RST}')
OK  = lambda s: print(f'    {GRN}{s}{RST}')
WRN = lambda s: print(f'    {YLW}{s}{RST}')
ERR = lambda s: print(f'    {RED}{s}{RST}')

# ────────────────────────────────────────────────────────────────────────────
for path in SESSION_PATHS:
    print(f'\n{BLD}{"="*72}{RST}')
    print(f'{BLD}  {path}{RST}')
    print(f'{BLD}{"="*72}{RST}')

    try:
        with NWBHDF5IO(path, 'r') as io:
            nwb = io.read()

            # ── 1. AIRES & NEURONES ──────────────────────────────────────────
            HDR('[1] Aires disponibles & nombre de neurones')
            units_df = nwb.units.to_dataframe()
            print(f'    Unités totales : {len(units_df)}')

            # Détecte la colonne d'aire selon le format
            for col_candidate in ('Target_area', 'ccf_parent_acronym', 'ccf_acronym',
                                  'brain_area', 'location'):
                if col_candidate in units_df.columns:
                    area_col = col_candidate
                    break
            else:
                area_col = None

            if area_col:
                print(f'    Colonne aire   : {area_col}')
                raw = units_df[area_col].values
                areas = np.array([a.decode() if isinstance(a, (bytes, bytearray)) else str(a)
                                   for a in raw])
                counts = Counter(areas)
                wS1_synonyms = {'SSp-bfd', 'wS1', 'BC', 'S1BF'}
                print(f'    {len(counts)} aire(s) :')
                for aire, n in sorted(counts.items(), key=lambda x: -x[1]):
                    marker = f'{GRN}★{RST}' if aire in wS1_synonyms or 'SSp' in aire else ' '
                    print(f'      {marker} {aire:35s} {n:3d} neurones')
                print(f'    (★ = probable wS1 / SSp-bfd)')
            else:
                ERR(f'Aucune colonne d\'aire trouvée.')
                print(f'    Colonnes dispo : {list(units_df.columns)}')

            # ── 2. STRUCTURE DES STIMULI ─────────────────────────────────────
            HDR('[2] Structure des stimuli')
            tr = nwb.trials.to_dataframe()
            print(f'    Trials totaux : {len(tr)}')

            # — Whisker stim —
            if 'whisker_stim_amplitude' in tr.columns:
                vals   = tr['whisker_stim_amplitude'].dropna()
                amps   = np.sort(vals[vals > 0].unique())
                n_whisk = len(vals[vals > 0])
                if len(amps) > 1:
                    print(f'\n    Whisker stim  : {GRN}{len(amps)} amplitudes{RST} → {amps}')
                else:
                    print(f'\n    Whisker stim  : {YLW}amplitude unique{RST} → {amps}')
                print(f'    Trials whisker : {n_whisk}')
                for a in amps:
                    n = int((tr['whisker_stim_amplitude'] == a).sum())
                    print(f'      amp {a:.0f} : {n:5d} trials')

            elif 'whisker_stim' in tr.columns:
                col = tr['whisker_stim']
                try:
                    n_w = int(col.astype(float).sum())
                except Exception:
                    n_w = col.notna().sum()
                print(f'\n    Whisker stim  : {YLW}binaire (0/1) — stimulus unique, pas d\'amplitude{RST}')
                print(f'    Trials whisker : {n_w}')

            elif 'whisker_stim_time' in tr.columns:
                n_w = tr['whisker_stim_time'].notna().sum()
                print(f'\n    Whisker stim  : colonne whisker_stim_time ({n_w} trials)')

            else:
                WRN('Aucune colonne whisker_stim trouvée.')

            # — Auditory stim —
            for aud_col in ('auditory_stim', 'auditory_stim_amplitude', 'auditory_stim_time'):
                if aud_col in tr.columns:
                    try:
                        n_a = int(tr[aud_col].astype(float).sum())
                    except Exception:
                        n_a = tr[aud_col].notna().sum()
                    print(f'\n    Auditory stim : {n_a} trials  (colonne \'{aud_col}\')')
                    break

            # — Catch / no-stim —
            for catch_col in ('no_stim', 'catch', 'no_stim_time'):
                if catch_col in tr.columns:
                    try:
                        n_c = int(tr[catch_col].astype(float).sum())
                    except Exception:
                        n_c = tr[catch_col].notna().sum()
                    print(f'    Catch         : {n_c} trials  (colonne \'{catch_col}\')')
                    break

            # — Résumé rapide des colonnes —
            print(f'\n    Toutes les colonnes trials : {list(tr.columns)}')

            # ── 3. LICK TIMES ────────────────────────────────────────────────
            HDR('[3] Lick times disponibles')
            try:
                beh = nwb.processing['behavior']
                found_any = False

                # PiezoLickSignal (signal continu)
                try:
                    ps = beh['BehavioralTimeSeries']['PiezoLickSignal']
                    sr = round(1.0 / float(np.median(np.diff(np.array(ps.timestamps[:500])))))
                    OK(f'PiezoLickSignal (continu) : {len(ps.data)} pts @ ~{sr} Hz')
                    found_any = True
                except Exception:
                    pass

                # BehavioralEvents
                try:
                    be = beh['BehavioralEvents']
                    for key in be.time_series:
                        obj = be[key]
                        ts  = np.array(obj.timestamps[:])
                        OK(f'BehavioralEvents[\'{key}\'] : {len(ts)} timestamps')
                        found_any = True
                except Exception:
                    pass

                if not found_any:
                    WRN('Aucun signal de lick trouvé dans behavior.')

            except Exception as e:
                ERR(f'Erreur accès behavior : {e}')

    except FileNotFoundError:
        ERR(f'FICHIER INTROUVABLE : {path}')
    except Exception as e:
        ERR(f'ERREUR LECTURE : {e}')

print(f'\n{BLD}{"="*72}{RST}')
print(f'{BLD}Vérification terminée — {len(SESSION_PATHS)} session(s).{RST}')




  data/brut/WR+/AO026_20181122_180943.nwb

[1] Aires disponibles & nombre de neurones
    Unités totales : 81
    Colonne aire   : Target_area
    2 aire(s) :
        mPFC                                 51 neurones
      ★ wS1                                  30 neurones
    (★ = probable wS1 / SSp-bfd)

[2] Structure des stimuli
    Trials totaux : 289

    Whisker stim  : 4 amplitudes → [1. 2. 3. 4.]
    Trials whisker : 146
      amp 1 :    38 trials
      amp 2 :    39 trials
      amp 3 :    36 trials
      amp 4 :    33 trials
    Catch         : 143 trials  (colonne 'no_stim')

    Toutes les colonnes trials : ['start_time', 'stop_time', 'trial_type', 'whisker_stim', 'whisker_stim_amplitude', 'whisker_stim_time', 'whisker_stim_duration', 'no_stim', 'no_stim_time', 'reward_available', 'response_window_start_time', 'response_window_stop_time', 'perf', 'lick_time', 'jaw_dlc_licks', 'lick_flag']

[3] Lick times disponibles
    PiezoLickSignal (continu) : 5094295 pts @ ~1000 Hz
   

In [15]:
# ── Filtre : sessions avec wS1/SSp-bfd ≥ 10 neurones + résumé stimuli ──────
# Critères :
#   ✓ wS1 présent (Target_area == 'wS1'  OU  ccf_parent_acronym == 'SSp-bfd'
#                  OU aire in {'BC', 'S1BF'})
#   ✓ ≥ 10 neurones dans cette aire
# Affiche aussi : type de stim (amp1-4 vs unique) + lick dispo

import numpy as np
from pynwb import NWBHDF5IO
from collections import Counter

BLD = '\033[1m'; GRN = '\033[92m'; YLW = '\033[93m'; RED = '\033[91m'; RST = '\033[0m'
WSS = {'SSp-bfd', 'wS1', 'BC', 'S1BF'}          # synonymes wS1
MIN_NEURONS = 10

valid, invalid = [], []

for path in SESSION_PATHS:
    name = path.split('/')[-1]
    group = os.path.basename(os.path.dirname(path))
    row  = {'path': path, 'name': name, 'group': group,
            'ws1_area': None, 'n_ws1': 0,
            'stim_type': '?', 'amps': [],
            'lick': '?', 'error': None}
    try:
        with NWBHDF5IO(path, 'r') as io:
            nwb = io.read()

            # ── Aires ────────────────────────────────────────────────────
            units_df = nwb.units.to_dataframe()
            for col in ('Target_area', 'ccf_parent_acronym', 'ccf_acronym', 'brain_area', 'location'):
                if col in units_df.columns:
                    area_col = col; break
            else:
                area_col = None

            if area_col:
                raw   = units_df[area_col].values
                areas = np.array([a.decode() if isinstance(a, (bytes, bytearray)) else str(a)
                                   for a in raw])
                counts = Counter(areas)
                # cherche la meilleure aire wS1
                for candidate in ('wS1', 'SSp-bfd', 'BC', 'S1BF'):
                    if candidate in counts:
                        row['ws1_area'] = candidate
                        row['n_ws1']    = counts[candidate]
                        break
                # fallback : toute aire contenant 'SSp'
                if row['ws1_area'] is None:
                    for aire, n in counts.items():
                        if 'SSp' in aire:
                            row['ws1_area'] = aire
                            row['n_ws1']    = n
                            break

            # ── Stimuli ──────────────────────────────────────────────────
            tr = nwb.trials.to_dataframe()
            if 'whisker_stim_amplitude' in tr.columns:
                vals = tr['whisker_stim_amplitude'].dropna()
                amps = np.sort(vals[vals > 0].unique())
                row['amps']      = list(amps.astype(int))
                row['stim_type'] = f'{len(amps)} amp(s)' if len(amps) > 1 else 'unique'
            elif 'whisker_stim' in tr.columns:
                row['stim_type'] = 'unique (binaire)'
                row['amps']      = [1]
            elif 'whisker_stim_time' in tr.columns:
                row['stim_type'] = 'unique (time only)'
                row['amps']      = [1]
            else:
                row['stim_type'] = 'INCONNU'

            # ── Lick ─────────────────────────────────────────────────────
            lick_src = []
            try:
                beh = nwb.processing['behavior']
                try:
                    beh['BehavioralTimeSeries']['PiezoLickSignal']
                    lick_src.append('Piezo')
                except Exception:
                    pass
                try:
                    be = beh['BehavioralEvents']
                    for k in be.time_series:
                        lick_src.append(k)
                except Exception:
                    pass
            except Exception:
                pass
            row['lick'] = ', '.join(lick_src) if lick_src else 'AUCUN'

    except FileNotFoundError:
        row['error'] = 'FICHIER INTROUVABLE'
    except Exception as e:
        row['error'] = str(e)[:60]

    if row['error']:
        invalid.append(row)
    elif row['n_ws1'] >= MIN_NEURONS:
        valid.append(row)
    else:
        invalid.append(row)

# ── Affichage ────────────────────────────────────────────────────────────────
print(f'\n{BLD}{"="*72}{RST}')
print(f'{BLD}  ✓ Sessions VALIDES  (wS1 ≥ {MIN_NEURONS} neurones) — {len(valid)}/{len(SESSION_PATHS)}{RST}')
print(f'{BLD}{"="*72}{RST}')

col_w = [35, 5, 12, 18, 10]
header = f"  {'Session':<{col_w[0]}} {'Grp':^{col_w[1]}} {'n wS1':>{col_w[2]}} {'Stim':^{col_w[3]}} {'Lick':<{col_w[4]}}"
print(f'{BLD}{header}{RST}')
print('  ' + '-'*(sum(col_w)+5))

for r in valid:
    aire_str  = f"{r['ws1_area']} ({r['n_ws1']})"
    stim_str  = r['stim_type']
    if r['amps'] and len(r['amps']) > 1:
        stim_str += f"  [{','.join(str(a) for a in r['amps'])}]"
    lick_ok   = 'AUCUN' not in r['lick']
    lick_str  = (GRN if lick_ok else RED) + r['lick'] + RST
    grp_str = r.get('group', '?')
    print(f"  {GRN}{r['name']:<{col_w[0]}}{RST} {grp_str:^{col_w[1]}} {aire_str:>{col_w[2]}}  {stim_str:<{col_w[3]}} {lick_str}")

if not valid:
    print(f'  {YLW}(aucune session ne passe le filtre){RST}')

print(f'\n{BLD}{"="*72}{RST}')
print(f'{BLD}  ✗ Sessions INVALIDES / ignorées — {len(invalid)}{RST}')
print(f'{BLD}{"="*72}{RST}')

for r in invalid:
    if r['error']:
        print(f"  {RED}✗ {r['name']:<40} {r['error']}{RST}")
    else:
        ws1_info = f"{r['ws1_area']} ({r['n_ws1']})" if r['ws1_area'] else 'pas de wS1'
        print(f"  {YLW}✗ {r['name']:<40} {ws1_info:<20} stim={r['stim_type']}{RST}")

# ── Chemins copiables pour results.ipynb ─────────────────────────────────────
if valid:
    print(f'\n{BLD}── SESSION_PATHS à copier dans results.ipynb ──{RST}')
    print('SESSION_PATHS = [')
    for r in valid:
        print(f"    '{r['path']}',")
    print(']')


  ✓ Sessions VALIDES  (wS1 ≥ 10 neurones) — 4/7
  Session                              Grp         n wS1        Stim        Lick      
  -------------------------------------------------------------------------------------
  AO026_20181122_180943.nwb            WR+      wS1 (30)  4 amp(s)  [1,2,3,4] Piezo, EngagedTrials, ReactionTimes, ResponseType, StimFlags, TrialOnsets, VideoOnsets, correct_rejection_trial, false_alarm_trial, jaw_dlc_licks, whisker_hit_trial, whisker_miss_trial
  AO081_20210505_192239.nwb            WR-      wS1 (36)  4 amp(s)  [1,2,3,4] ResponseType, Reward_Window_onset, Reward_time, StimFlags, Valve_Ind_Assosiation, Valve_Ind_MouseTriggered, jaw_dlc_licks, whisker_hit_trial, whisker_miss_trial
  AO082_20210519_183538.nwb            WR-      wS1 (48)  4 amp(s)  [1,2,3,4] ResponseType, Reward_Window_onset, Reward_time, StimFlags, Valve_Ind_Assosiation, Valve_Ind_MouseTriggered, jaw_dlc_licks, whisker_hit_trial, whisker_miss_trial
  AO082_20210521_164409.nwb        

In [16]:
# ── Tri : valid → data/valid/WR+|WR-/   |   non valide → data/non_valid/WR+|WR-/ ──
import shutil, os

VALID_BASE   = 'data/valid'
INVALID_BASE = 'data/non_valid'
for grp in ['WR+', 'WR-']:
    os.makedirs(os.path.join(VALID_BASE,   grp), exist_ok=True)
    os.makedirs(os.path.join(INVALID_BASE, grp), exist_ok=True)

BLD = '\033[1m'; GRN = '\033[92m'; YLW = '\033[93m'; RED = '\033[91m'; RST = '\033[0m'

def move_file(src, dest_dir, label_color):
    name = os.path.basename(src)
    dst  = os.path.join(dest_dir, name)
    if not os.path.exists(src):
        print(f'  {RED}x introuvable  {name}{RST}')
        return 'error'
    if os.path.exists(dst):
        print(f'  {YLW}o deja present {name}  ->  {dest_dir}/{RST}')
        return 'skip'
    try:
        shutil.move(src, dst)
        print(f'  {label_color}v deplace      {name}  ->  {dest_dir}/{RST}')
        return 'ok'
    except Exception as e:
        print(f'  {RED}x erreur       {name}  ->  {e}{RST}')
        return 'error'

print(f'{BLD}-- Sessions VALIDES --{RST}')
v_ok = v_skip = v_err = 0
for r in valid:
    dest = os.path.join(VALID_BASE, r.get('group', 'WR+'))
    res  = move_file(r['path'], dest, GRN)
    if res == 'ok':     v_ok   += 1
    elif res == 'skip': v_skip += 1
    else:               v_err  += 1

print(f'\n{BLD}-- Sessions INVALIDES --{RST}')
i_ok = i_skip = i_err = 0
for r in invalid:
    if r['error'] and 'INTROUVABLE' in str(r['error']):
        continue
    dest = os.path.join(INVALID_BASE, r.get('group', 'WR+'))
    res  = move_file(r['path'], dest, YLW)
    if res == 'ok':     i_ok   += 1
    elif res == 'skip': i_skip += 1
    else:               i_err  += 1

print(f'\n{BLD}--- Resume ---{RST}')
print(f'  -> {VALID_BASE}/   : {GRN}{v_ok} deplace(s){RST}  {YLW}{v_skip} deja present(s){RST}  {RED}{v_err} erreur(s){RST}')
print(f'  -> {INVALID_BASE}/ : {YLW}{i_ok} deplace(s){RST}  {YLW}{i_skip} deja present(s){RST}  {RED}{i_err} erreur(s){RST}')

# ── SESSION_PATHS prêts pour results.ipynb ─────────────────────────────────
print(f'\n{BLD}SESSION_PATHS pour results.ipynb :{RST}')
for grp in ['WR+', 'WR-']:
    grp_dir = os.path.join(VALID_BASE, grp)
    if not os.path.isdir(grp_dir): continue
    files = sorted([os.path.join(grp_dir, f)
                    for f in os.listdir(grp_dir) if f.endswith('.nwb')])
    if files:
        print(f'  # {grp} ({len(files)} sessions)')
        for p in files:
            print(f"    '{p}',")


-- Sessions VALIDES --
  v deplace      AO026_20181122_180943.nwb  ->  data/valid/WR+/
  v deplace      AO081_20210505_192239.nwb  ->  data/valid/WR-/
  v deplace      AO082_20210519_183538.nwb  ->  data/valid/WR-/
  v deplace      AO082_20210521_164409.nwb  ->  data/valid/WR-/

-- Sessions INVALIDES --
  v deplace      AO081_20210506_132701.nwb  ->  data/non_valid/WR-/
  v deplace      AO081_20210507_131703.nwb  ->  data/non_valid/WR-/
  v deplace      AO082_20210520_160848.nwb  ->  data/non_valid/WR-/

--- Resume ---
  -> data/valid/   : 4 deplace(s)  0 deja present(s)  0 erreur(s)
  -> data/non_valid/ : 3 deplace(s)  0 deja present(s)  0 erreur(s)

SESSION_PATHS pour results.ipynb :
  # WR+ (22 sessions)
    'data/valid/WR+/AO026_20181122_180943.nwb',
    'data/valid/WR+/AO026_20181123_110337.nwb',
    'data/valid/WR+/AO027_20181101_134512.nwb',
    'data/valid/WR+/AO028_20181102_110339.nwb',
    'data/valid/WR+/AO028_20181103_83222.nwb',
    'data/valid/WR+/AO036_20190507_183647.nw

In [17]:
# ── EngagedTrials : pattern analysis — does disengagement persist? ─────────
import glob, os, numpy as np
from pynwb import NWBHDF5IO

VALID_BASE  = 'data/valid'
valid_paths = sorted([
    p for grp in ['WR+', 'WR-']
    for p in glob.glob(os.path.join(VALID_BASE, grp, '*.nwb'))
])

for path in valid_paths:
    name = os.path.basename(path)
    with NWBHDF5IO(path, 'r') as io:
        nwb = io.read()
        try:
            et   = nwb.processing['behavior']['BehavioralEvents']['EngagedTrials']
            ts   = np.array(et.timestamps[:])
            data = np.array(et.data[:])
        except Exception:
            print(f"{name}: no EngagedTrials\n"); continue

    # Transitions
    transitions = np.diff(data.astype(int))
    engage_to_dis = np.where(transitions == -1)[0]   # 1→0
    dis_to_engage = np.where(transitions ==  1)[0]   # 0→1

    first_zero = np.where(data == 0)[0]
    if len(first_zero) == 0:
        status = "always engaged ✓"
        last_engaged_t = ts[-1]
    else:
        first_zero_idx = first_zero[0]
        # Any 1 after the first 0?
        re_engagements = np.where(data[first_zero_idx:] == 1)[0]
        if len(re_engagements) == 0:
            status = f"disengaged at t={ts[first_zero_idx]:.1f}s — stays OFF ✗"
            last_engaged_t = ts[first_zero_idx - 1] if first_zero_idx > 0 else 0
        else:
            status = f"disengaged at t={ts[first_zero_idx]:.1f}s — RE-ENGAGES later ↺"
            last_engaged_t = None

    n_engaged   = np.sum(data == 1)
    n_disengaged = np.sum(data == 0)
    pct_engaged = 100 * n_engaged / len(data)

    print(f"{name}")
    print(f"  Status : {status}")
    print(f"  Engaged trials   : {n_engaged}/{len(data)} ({pct_engaged:.0f}%)")
    print(f"  Disengaged trials: {n_disengaged}")
    if len(dis_to_engage) > 0:
        print(f"  Re-engagement times : {np.round(ts[dis_to_engage+1], 1)}")
    if last_engaged_t is not None:
        print(f"  Last engaged t  : {last_engaged_t:.1f}s  (session ends {ts[-1]:.1f}s)")
    print()

AO026_20181122_180943.nwb
  Status : always engaged ✓
  Engaged trials   : 289/289 (100%)
  Disengaged trials: 0
  Last engaged t  : 3880.9s  (session ends 3880.9s)

AO026_20181123_110337.nwb
  Status : disengaged at t=4883.8s — stays OFF ✗
  Engaged trials   : 394/456 (86%)
  Disengaged trials: 62
  Last engaged t  : 4875.9s  (session ends 5641.4s)

AO027_20181101_134512.nwb
  Status : disengaged at t=4643.9s — stays OFF ✗
  Engaged trials   : 345/402 (86%)
  Disengaged trials: 57
  Last engaged t  : 4628.9s  (session ends 5381.6s)

AO028_20181102_110339.nwb
  Status : always engaged ✓
  Engaged trials   : 446/446 (100%)
  Disengaged trials: 0
  Last engaged t  : 6502.8s  (session ends 6502.8s)

AO028_20181103_83222.nwb
  Status : always engaged ✓
  Engaged trials   : 408/408 (100%)
  Disengaged trials: 0
  Last engaged t  : 5767.6s  (session ends 5767.6s)

AO036_20190507_183647.nwb
  Status : always engaged ✓
  Engaged trials   : 849/849 (100%)
  Disengaged trials: 0
  Last engaged t